# AI-Generated Face Detection Using Spatial and Frequency Domain Features
**Late fusion of a spatial branch (ResNet-18) and a frequency branch (FFT + small CNN), tested on unseen generators.**

Team: Lokesh Kumar, Dhruv Khurana, Sameer Yadav | Guide: Dr. Sudhir Dawra | SRM Institute of Science and Technology, Delhi-NCR, Ghaziabad

**How to use this notebook**
1. Build the dataset subset first on Kaggle with `kaggle_build_dataset.py` (see README.md). It gives `data/trainval`, `data/test_unseen` and `data/test_unseen2`.
2. Put the extracted `data` folder next to this notebook, open it with the Python 3.12 + PyTorch (CUDA) environment, and run the cells from top to bottom.
3. Training takes about 17 minutes for the 3-seed run on an RTX 3050. Run the training cell only once, then reuse the saved files.

Folder layout expected next to this notebook: `data/trainval/{real,fake}`, `data/test_unseen/{real,fake}`, `data/test_unseen2/{real,fake}`.

## 1. Check the GPU

In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 2. Imports, file lists and the train / validation split

In [ ]:
import os, glob, json, time, pickle, numpy as np, torch, torch.nn as nn, torchvision
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

dev = "cuda"
torch.manual_seed(42); np.random.seed(42)

def files(split):
    r = sorted(glob.glob(f"data/{split}/real/*.jpg")); f = sorted(glob.glob(f"data/{split}/fake/*.jpg"))
    return r + f, [0]*len(r) + [1]*len(f)          # label 0 = real, 1 = fake

X, y = files("trainval")
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
Xte, yte = files("test_unseen")                    # unseen generator 1: GANformer
for s in ("trainval", "test_unseen", "test_unseen2"):
    for l in ("real", "fake"):
        print(s, l, len(glob.glob(f"data/{s}/{l}/*.jpg")))

## 3. First look: average FFT spectrum of real vs fake faces
Saves `fft_real_vs_fake.png` (200 random real faces vs 200 random fake faces from the training set).

In [ ]:
import random
import matplotlib.pyplot as plt

def fft_mag(path):
    g = np.array(Image.open(path).convert("L").resize((200, 200)), dtype=np.float32)
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))

random.seed(0)
rf = random.sample(sorted(glob.glob("data/trainval/real/*.jpg")), 200)
ff = random.sample(sorted(glob.glob("data/trainval/fake/*.jpg")), 200)
real = np.mean([fft_mag(f) for f in rf], 0); fake = np.mean([fft_mag(f) for f in ff], 0)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, im, t in zip(ax, [real, fake, fake - real], ["Real (avg FFT)", "Fake (avg FFT)", "Fake - Real"]):
    h = a.imshow(im, cmap="viridis"); a.set_title(t); a.axis("off"); plt.colorbar(h, ax=a, fraction=0.046)
plt.tight_layout(); plt.savefig("fft_real_vs_fake.png", dpi=200); plt.show()

## 4. Data loaders
Each item returns the image (for the spatial branch) and its standardized FFT spectrum (for the frequency branch). The spectrum is standardized per frequency bin using the mean and standard deviation of 400 training images.

In [ ]:
norm = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def spec_of(path):
    g = np.array(Image.open(path).convert("L").resize((200, 200)), dtype=np.float32)
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))

idx = np.random.RandomState(0).choice(len(Xtr), 400, replace=False)
S = np.stack([spec_of(Xtr[i]) for i in idx]); MU, SD = S.mean(0), S.std(0) + 1e-6

class DS2(Dataset):
    def __init__(s, paths, labels, train=False): s.p, s.l, s.train = paths, labels, train
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        im = Image.open(s.p[i]).convert("RGB").resize((200, 200))
        if s.train and np.random.rand() < 0.5: im = im.transpose(Image.FLIP_LEFT_RIGHT)
        g = np.array(im.convert("L"), dtype=np.float32)
        sp = (np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g)))) - MU) / SD
        return norm(im), torch.from_numpy(sp).unsqueeze(0).float(), s.l[i]

tr_dl = DataLoader(DS2(Xtr, ytr, True), batch_size=32, shuffle=True,  num_workers=0)
va_dl = DataLoader(DS2(Xva, yva),       batch_size=64, shuffle=False, num_workers=0)
te_dl = DataLoader(DS2(Xte, yte),       batch_size=64, shuffle=False, num_workers=0)
print("setup OK | train", len(Xtr), "| val", len(Xva), "| unseen", len(Xte))

## 5. Models
- **Spatial**: ResNet-18 pretrained on ImageNet, last layer replaced by 2 outputs.
- **Freq**: small CNN on the FFT spectrum (about 0.1 M parameters).
- **Late fusion** (the proposed method) is done at test time by averaging the two probabilities, so it needs no extra class.

In [ ]:
def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

class FreqNet(nn.Module):
    def __init__(s):
        super().__init__()
        s.f = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128),
                            nn.AdaptiveAvgPool2d(1), nn.Flatten())
    def forward(s, x): return s.f(x)

class Spatial(nn.Module):
    def __init__(s):
        super().__init__()
        s.net = torchvision.models.resnet18(weights="IMAGENET1K_V1"); s.net.fc = nn.Linear(512, 2)
    def forward(s, im, sp): return s.net(im)

class Freq(nn.Module):
    def __init__(s):
        super().__init__()
        s.f = FreqNet(); s.fc = nn.Linear(128, 2)
    def forward(s, im, sp): return s.fc(s.f(sp))

## 6. Training and evaluation helpers
`fit` trains a model, keeps the epoch with the best validation accuracy, saves `<name>.pt` and `results_<name>.json`.

In [ ]:
@torch.no_grad()
def probs(model, dl):
    model.eval(); P, Y = [], []
    for im, sp, lb in dl:
        with torch.autocast("cuda"):
            out = model(im.to(dev), sp.to(dev))
        P.append(torch.softmax(out.float(), 1)[:, 1].cpu()); Y.append(lb)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def metrics(p, y):
    pr = (p > 0.5).astype(int)
    return accuracy_score(y, pr), f1_score(y, pr), roc_auc_score(y, p)

@torch.no_grad()
def evaluate(model, dl):
    p, t = probs(model, dl); pr = (p > 0.5).astype(int)
    return dict(acc=float(accuracy_score(t, pr)), precision=float(precision_score(t, pr)),
                recall=float(recall_score(t, pr)), f1=float(f1_score(t, pr)),
                auc=float(roc_auc_score(t, p)), cm=confusion_matrix(t, pr).tolist())

def fit(model, name, epochs=5, lr=1e-4):
    model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda"); lossf = nn.CrossEntropyLoss()
    best, best_state = 0, None
    for ep in range(epochs):
        model.train(); t0 = time.time(); tl = 0
        for im, sp, lb in tr_dl:
            im, sp, lb = im.to(dev), sp.to(dev), lb.to(dev)
            opt.zero_grad()
            with torch.autocast("cuda"):
                loss = lossf(model(im, sp), lb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += loss.item()
        m = evaluate(model, va_dl)
        print(f"epoch {ep+1}/{epochs} | loss {tl/len(tr_dl):.4f} | val acc {m['acc']:.4f} | {time.time()-t0:.0f}s")
        if m["acc"] > best:
            best = m["acc"]; best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); torch.save(best_state, f"{name}.pt")
    res = {"val_seen": evaluate(model, va_dl), "test_unseen": evaluate(model, te_dl)}
    json.dump(res, open(f"results_{name}.json", "w"), indent=1)
    return res

## 7. Train spatial and frequency models with 3 seeds (about 17 minutes)
Saves `spatial_s1..3.pt`, `frequency_s1..3.pt` and all predictions in `seed_probs.pkl`. Run once.

In [ ]:
SEEDS = [1, 2, 3]
store = {}      # (model, seed, split) -> (probabilities, labels)

for seed in SEEDS:
    for name, make, ep, lr in [("spatial", Spatial, 5, 1e-4), ("frequency", Freq, 15, 1e-3)]:
        torch.manual_seed(seed); np.random.seed(seed)
        print(f"=== {name} | seed {seed} ===")
        m = make()
        fit(m, f"{name}_s{seed}", epochs=ep, lr=lr)
        for split, dl in [("val", va_dl), ("test", te_dl)]:
            store[(name, seed, split)] = probs(m, dl)
        pickle.dump(store, open("seed_probs.pkl", "wb"))
print("ALL DONE")

## 8. Results table: spatial, frequency and late fusion (mean ± std over 3 seeds)
Late fusion = 0.5 x spatial probability + 0.5 x frequency probability. The weight 0.5 was fixed in advance, not tuned.

In [ ]:
store = pickle.load(open("seed_probs.pkl", "rb"))
SEEDS = [1, 2, 3]

def get(name): return lambda s, sp: store[(name, s, sp)]
def late(s, sp):
    ps, y = store[("spatial", s, sp)]; pf, _ = store[("frequency", s, sp)]
    return 0.5 * ps + 0.5 * pf, y

for label, fn in [("Spatial", get("spatial")), ("Frequency", get("frequency")), ("Late fusion", late)]:
    for split, nm in (("val", "seen"), ("test", "unseen GANformer")):
        ms = np.array([metrics(*fn(s, split)) for s in SEEDS]); m, sd = ms.mean(0), ms.std(0)
        print(f"{label:12s} {nm:17s} acc {m[0]*100:.1f}±{sd[0]*100:.1f} | F1 {m[1]:.3f}±{sd[1]:.3f} | AUC {m[2]:.3f}±{sd[2]:.3f}")

## 9. Second unseen generator (CIPS)
Needs `data/test_unseen2` (built by the second part of `kaggle_build_dataset.py`). Uses the saved weights only, no training.

In [ ]:
X2, y2 = files("test_unseen2")
dl2 = DataLoader(DS2(X2, y2), batch_size=64, shuffle=False, num_workers=0)
print("CIPS test:", len(X2), "images")

store2 = {}
for seed in SEEDS:
    for name, make in [("spatial", Spatial), ("frequency", Freq)]:
        m = make().to(dev); m.load_state_dict(torch.load(f"{name}_s{seed}.pt"))
        store2[(name, seed)] = probs(m, dl2)
pickle.dump(store2, open("cips_probs.pkl", "wb"))

def late2(s):
    ps, y = store2[("spatial", s)]; pf, _ = store2[("frequency", s)]
    return 0.5 * ps + 0.5 * pf, y

for label, fn in [("Spatial", lambda s: store2[("spatial", s)]),
                  ("Frequency", lambda s: store2[("frequency", s)]),
                  ("Late fusion", late2)]:
    ms = np.array([metrics(*fn(s)) for s in SEEDS]); m_, sd = ms.mean(0), ms.std(0)
    print(f"{label:12s} CIPS acc {m_[0]*100:.1f}±{sd[0]*100:.1f} | F1 {m_[1]:.3f}±{sd[1]:.3f} | AUC {m_[2]:.3f}±{sd[2]:.3f}")

## 10. Final results table and accuracy chart
Saves `results_all.csv` and `fig_accuracy_all.png`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

store  = pickle.load(open("seed_probs.pkl", "rb"))
store2 = pickle.load(open("cips_probs.pkl", "rb"))
SEEDS = [1, 2, 3]; models = ["Spatial", "Frequency", "Late fusion"]
conds = ["Seen", "Unseen: GANformer", "Unseen: CIPS"]

def P(m, s, c):
    src = {"Seen": lambda n: store[(n, s, "val")],
           "Unseen: GANformer": lambda n: store[(n, s, "test")],
           "Unseen: CIPS": lambda n: store2[(n, s)]}[c]
    if m == "Late fusion":
        ps, y = src("spatial"); pf, _ = src("frequency"); return 0.5 * ps + 0.5 * pf, y
    return src(m.lower())

rows = []; acc_m = {}; acc_s = {}
for m in models:
    for c in conds:
        vals = []
        for s in SEEDS:
            p, y = P(m, s, c); pr = (p > 0.5).astype(int)
            vals.append([accuracy_score(y, pr) * 100, f1_score(y, pr), roc_auc_score(y, p)])
        v = np.array(vals); acc_m[(m, c)], acc_s[(m, c)] = v[:, 0].mean(), v[:, 0].std()
        rows.append(dict(model=m, test_set=c, acc=f"{v[:,0].mean():.1f}±{v[:,0].std():.1f}",
                         f1=f"{v[:,1].mean():.3f}±{v[:,1].std():.3f}", auc=f"{v[:,2].mean():.3f}±{v[:,2].std():.3f}"))
tbl = pd.DataFrame(rows); tbl.to_csv("results_all.csv", index=False); print(tbl.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.3)); w = 0.26
for k, c in enumerate(conds):
    ax.bar(np.arange(3) + (k - 1) * w, [acc_m[(m, c)] for m in models], w,
           yerr=[acc_s[(m, c)] for m in models], capsize=3, label=c)
ax.set_xticks(range(3)); ax.set_xticklabels(models); ax.set_ylabel("Accuracy (%)"); ax.set_ylim(50, 95); ax.legend()
plt.tight_layout(); plt.savefig("fig_accuracy_all.png", dpi=200); plt.show()

## 11. ROC curves and confusion matrices
Saves `fig_roc.png` and `fig_confusion_unseen.png`. Runs after the previous cell (it reuses `P`, `models`, `conds`).

In [ ]:
from sklearn.metrics import roc_curve
colors = {"Spatial": "tab:blue", "Frequency": "tab:orange", "Late fusion": "tab:green"}

grid = np.linspace(0, 1, 200)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, c in zip(ax, conds):
    for m in models:
        tprs, aucs = [], []
        for s in SEEDS:
            p, y = P(m, s, c); f, t, _ = roc_curve(y, p)
            tprs.append(np.interp(grid, f, t)); aucs.append(roc_auc_score(y, p))
        a.plot(grid, np.mean(tprs, 0), color=colors[m], label=f"{m} (AUC {np.mean(aucs):.3f})")
    a.plot([0, 1], [0, 1], "k--", lw=0.8); a.set_title(c)
    a.set_xlabel("False positive rate"); a.set_ylabel("True positive rate"); a.legend(loc="lower right")
plt.tight_layout(); plt.savefig("fig_roc.png", dpi=200); plt.show()

fig, ax = plt.subplots(2, 3, figsize=(12, 7.5))
for r, c in enumerate(conds[1:]):
    for k, m in enumerate(models):
        cm = sum(confusion_matrix(P(m, s, c)[1], (P(m, s, c)[0] > 0.5).astype(int)) for s in SEEDS).astype(float)
        cmn = cm / cm.sum(1, keepdims=True); a = ax[r, k]
        a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cmn[i,j]*100:.1f}%", ha="center", va="center",
                       color="white" if cmn[i, j] > 0.5 else "black")
        a.set_xticks([0, 1]); a.set_xticklabels(["Real", "Fake"]); a.set_yticks([0, 1]); a.set_yticklabels(["Real", "Fake"])
        a.set_xlabel("Predicted"); a.set_ylabel("Actual"); a.set_title(f"{m} | {c}")
plt.tight_layout(); plt.savefig("fig_confusion_unseen.png", dpi=200); plt.show()

## 12. Optional baseline: joint fusion (both branches trained together)
This is the baseline that generalized worse in the paper. **Note:** the joint-fusion numbers in the paper come from single runs made during development (an earlier version used per-image spectrum normalization). Running this cell gives a fresh single run with the current loaders, so the numbers can differ a little.

In [ ]:
class Hybrid2(nn.Module):
    def __init__(s):
        super().__init__()
        s.s = torchvision.models.resnet18(weights="IMAGENET1K_V1"); s.s.fc = nn.Identity()
        s.f = FreqNet()
        s.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(512 + 128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(s, im, sp):
        a, b = s.s(im), s.f(sp)
        if s.training:                                   # randomly hide one branch while training
            r = torch.rand(a.size(0), 1, device=a.device)
            a = a * (r >= 0.15).float()
            b = b * ((r < 0.15) | (r >= 0.30)).float()
        return s.head(torch.cat([a, b], 1))

torch.manual_seed(1); np.random.seed(1)
res_h = fit(Hybrid2(), "hybrid2", epochs=8, lr=2e-4)
for k, v in res_h.items():
    print(k, {a: round(b, 4) for a, b in v.items() if a != "cm"})

## 13. Save the normalization statistics (needed by the Streamlit app)

In [ ]:
np.savez("norm_stats.npz", MU=MU, SD=SD)
print("saved norm_stats.npz")

## 14. Demo images for the app
Copies 5 confident CIPS fakes, 5 confident GANformer fakes, 5 confident real faces and 3 misses per generator into `demo_images/`. The number in each file name is the model's fake-probability. Show a miss too when presenting: the model is not perfect.

In [ ]:
import shutil
st_ = np.load("norm_stats.npz"); MU2, SD2 = st_["MU"], st_["SD"]
a = Spatial().to(dev); a.load_state_dict(torch.load("spatial_s1.pt")); a.eval()
b = Freq().to(dev);    b.load_state_dict(torch.load("frequency_s1.pt")); b.eval()

def app_predict(path):
    im = Image.open(path).convert("RGB").resize((200, 200))
    g = np.array(im.convert("L"), dtype=np.float32)
    raw = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))
    sp = torch.from_numpy((raw - MU2) / SD2).unsqueeze(0).unsqueeze(0).float().to(dev)
    x = norm(im).unsqueeze(0).to(dev)
    with torch.no_grad():
        ps = torch.softmax(a(x, sp), 1)[0, 1].item(); pf = torch.softmax(b(x, sp), 1)[0, 1].item()
    return ps, pf, 0.5 * ps + 0.5 * pf

os.makedirs("demo_images", exist_ok=True)
def pick(folder, tag, n=5, fake=True):
    fs = sorted(glob.glob(folder + "/*.jpg"))[:300]
    sc = sorted(((app_predict(f)[2], f) for f in fs), reverse=fake)[:n]
    for p, f in sc: shutil.copy(f, f"demo_images/{tag}_{p:.2f}_{os.path.basename(f)}")
def pick_miss(folder, tag, n=3):
    fs = sorted(glob.glob(folder + "/*.jpg"))[:300]
    sc = sorted((app_predict(f)[2], f) for f in fs)[:n]
    for p, f in sc: shutil.copy(f, f"demo_images/{tag}_{p:.2f}_{os.path.basename(f)}")

pick("data/test_unseen2/fake", "FAKE_cips"); pick("data/test_unseen/fake", "FAKE_ganformer")
pick("data/test_unseen/real", "REAL", fake=False)
pick_miss("data/test_unseen/fake", "MISS_ganformer"); pick_miss("data/test_unseen2/fake", "MISS_cips")
print(sorted(os.listdir("demo_images")))

## 15. Run the demo app
In a new Command Prompt (not in this notebook): `cd` to this folder, activate the environment, then run `streamlit run app.py`.
The app needs these files next to it: `spatial_s1.pt`, `frequency_s1.pt`, `norm_stats.npz` (and optionally `demo_images/` and the four figure PNGs for the About page).